# Remote machines

linopy can ship your model to a remote machine, run a solver there, and pull the solved model back. Two remotes are supported:

- **SSH** — connect to a server you own (or have access to) over SSH.
- **OETC** — submit jobs to [OET Cloud](https://open-energy-transition.org/), a managed optimization service.

Both share the same entry point on `Model.solve`:

```python
m.solve("gurobi", remote=<Settings>, **solver_options)
```

`solver_name` and `**solver_options` work exactly like a local solve; `remote=` selects *where* to run. After the call, `model.remote` holds the remote instance for post-solve introspection (mirrors `model.solver`).

> **Note:** This notebook is not executed during the documentation build — it requires either SSH access to a remote server or OETC credentials.

## Create a model

Build the model locally as usual:

In [1]:
from numpy import arange
from xarray import DataArray

from linopy import Model

N = 10
m = Model()
coords = [arange(N), arange(N)]
x = m.add_variables(coords=coords, name="x")
y = m.add_variables(coords=coords, name="y")
m.add_constraints(x - y >= DataArray(arange(N)))
m.add_constraints(x + y >= 0)
m.add_objective((2 * x + y).sum())
m

Linopy LP model

Variables:
----------
 * x (dim_0, dim_1)
 * y (dim_0, dim_1)

Constraints:
------------
 * con0 (dim_0, dim_1)
 * con1 (dim_0, dim_1)

Status:
-------
initialized

## SSH

**What you need**

- `uv pip install "linopy[ssh]"` locally (pulls in `paramiko`).
- A remote server with linopy and a solver installed (e.g. in a conda environment).
- SSH access to that machine (key-based auth recommended).

Build an `SshSettings` and pass it as `remote=`. Use `setup_commands` to activate environments or export variables on the remote shell before the solve.

In [2]:
from linopy.remote import SshSettings

ssh_settings = SshSettings(
    hostname="your.host.de",
    username="username",
    # password="...",  # not needed when SSH keys are autodetected
    setup_commands=["conda activate linopy-env"],
)

m.solve("gurobi", remote=ssh_settings)
m.solution

gaierror: [Errno 8] nodename nor servname provided, or not known

## OETC

**What you need**

- `uv pip install "linopy[oetc]"` locally (pulls in `google-cloud-storage` and `requests`).
- An OETC account with valid credentials.
- The OETC authentication and orchestrator server URLs.

Build an `OetcSettings`. Two construction styles:

1. **Manually** — pass `email`, `password`, `name`, and the server URLs.
2. **`OetcSettings.from_env()`** — resolve everything from environment variables (`OETC_EMAIL`, `OETC_PASSWORD`, `OETC_NAME`, `OETC_AUTH_URL`, `OETC_ORCHESTRATOR_URL`). Recommended for CI/CD. Keyword arguments override the environment.

linopy uploads the model to OETC, submits a compute job, polls until it finishes, and downloads the solution — all behind one call.

In [ ]:
from linopy.remote import OetcSettings

# Option 1: pass credentials directly
oetc_settings = OetcSettings(
    email="your-email@example.com",
    password="your-password",
    name="linopy-example-job",
    authentication_server_url="https://auth.oetcloud.com",
    orchestrator_server_url="https://orchestrator.oetcloud.com",
    cpu_cores=4,
    disk_space_gb=20,
)

# Option 2: load from environment (with optional overrides)
# oetc_settings = OetcSettings.from_env(cpu_cores=4, disk_space_gb=20)

m.solve("gurobi", remote=oetc_settings, TimeLimit=600, MIPGap=0.01)

print(f"Status: {m.status}")
print(f"Objective: {m.objective.value:.4f}")
m.solution

## Advanced: drive the remote directly

For finer control — inspecting the round-tripped solved model, splitting submit from collect for async workflows — use the `Oetc` or `SSH` class directly. `Model.solve(remote=...)` runs the same path internally and then writes the result back onto the local model in place.

### SSH

In [ ]:
from linopy.remote import SSH

ssh = SSH(ssh_settings)
solved = ssh.solve(m, "gurobi", presolve="on")
solved.solution

### OETC

`Oetc` is a *connection*, not a job — authenticate once, then submit and collect any number of jobs:

1. `submit(model, solver_name, **options)` — upload and dispatch; returns a job uuid.
2. `status(job_uuid)` — a single, non-blocking status check.
3. `collect(job_uuid)` — wait for completion, download, and return the solved model.

A job is identified solely by its uuid string, so the lifecycle is async-friendly: submit many models on one connection, hold their uuids, and collect each when convenient — even from a different process (a fresh `Oetc(settings)` re-authenticates and collects by uuid).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from linopy.remote import Oetc

oetc = Oetc(oetc_settings)

# A single job: submit now, collect by uuid later (even in another process).
job_uuid = oetc.submit(m, "gurobi", TimeLimit=600, MIPGap=0.01)
print(f"Submitted job {job_uuid}")
solved = oetc.collect(job_uuid)
print(f"Status: {solved.status}")

# Many models on one connection: submit all, then collect concurrently.
models = [m]  # replace with your own models
uuids = [oetc.submit(model, "gurobi") for model in models]
with ThreadPoolExecutor() as pool:
    solved_models = list(pool.map(oetc.collect, uuids))